# TAB-R1: Cloud & Google Colab Foundation Model Evaluator

This notebook evaluates **Tabular Foundation Models (TabPFN v2, v2.5, v2.6, v3, Google TabFM)** and **AutoGluon** on the **400 frozen leakage-safe cancer multiomics folds**.

### Features:
- Runs on **Free Google Colab T4 GPU** or cloud VMs (GCP/AWS/Lambda)
- Auto-resumes from checkpoints if interrupted or disconnected
- Batch execution controls (`START_FOLD`, `END_FOLD`, `MODELS`)
- Saves sample-level test predictions and consolidated metrics matching the publication schema.

## 1. System Check & GPU Verification

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Install Dependencies & Clone Repository

In [ ]:
# Clone the repository if not already in workspace
import os
if not os.path.exists("tab-r1"):
    !git clone https://github.com/HawkFranklin-Research/tab-r1.git
    %cd tab-r1
else:
    %cd tab-r1
    !git pull

# Install dependencies
!pip install -q "tabpfn>=8.0.0" "autogluon.tabular>=1.1.0" huggingface_hub pandas scikit-learn lightgbm xgboost catboost

## 3. (Optional) Mount Google Drive for Automatic Checkpoint Backup

In [ ]:
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
OUTPUT_DIR = "/content/drive/MyDrive/tabr1_cloud_outputs" if USE_GOOGLE_DRIVE else "./cloud_outputs"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Outputs will be saved directly to Drive: {OUTPUT_DIR}")

## 4. Run Model Evaluation on Frozen Folds

Choose the models and fold slice to evaluate:

In [ ]:
# @title Evaluation Configuration
MODELS = "tabpfn_v3,tabpfn_v2_5,tabfm_default" #@param ["tabpfn_v3", "tabpfn_v2_5", "tabpfn_v2", "tabpfn_v2_6", "tabfm_default", "autogluon", "tabpfn_v3,tabpfn_v2_5,tabfm_default", "all_foundation"] {allow-input: true}
START_FOLD = 0 #@param {type:"integer"}
END_FOLD = 50 #@param {type:"integer"}
DEVICE = "auto" #@param ["auto", "cuda", "cpu"]
TABFM_BACKEND = "pytorch" #@param ["pytorch", "jax"]
AUTOGLUON_TIME_LIMIT = 180 #@param {type:"integer"}

if MODELS == "all_foundation":
    MODELS = "tabpfn_v2,tabpfn_v2_5,tabpfn_v2_6,tabpfn_v3,tabfm_default"

!python paper/analysis/run_cloud_evaluation.py \
  --models "{MODELS}" \
  --start-fold {START_FOLD} \
  --end-fold {END_FOLD} \
  --device "{DEVICE}" \
  --tabfm-backend "{TABFM_BACKEND}" \
  --autogluon-time-limit {AUTOGLUON_TIME_LIMIT} \
  --output-root "{OUTPUT_DIR}" \
  --resume

## 5. View Evaluation Results Summary

In [ ]:
import pandas as pd
from pathlib import Path

metrics_file = Path(OUTPUT_DIR) / "all_fold_model_metrics.csv"
if metrics_file.exists():
    df = pd.read_csv(metrics_file)
    print(f"Total evaluated fold runs: {len(df)}")
    summary = df[df["status"] == "success"].groupby(["model_name", "endpoint"])[["roc_auc", "pr_auc", "f1", "balanced_accuracy", "log_loss"]].mean().reset_index()
    display(summary)
else:
    print("No metrics file found at:", metrics_file)

## 6. Compress and Download Results

In [ ]:
import shutil
from google.colab import files

zip_filename = "tabr1_cloud_predictions.zip"
shutil.make_archive("tabr1_cloud_predictions", "zip", OUTPUT_DIR)
print(f"Archived results to {zip_filename}")
files.download(zip_filename)